# SECP256k1 — Toy Wallet & Ethereum Address Generation

<style>
    .python-function {
        color: #228B22;
        font-family: monospace;
    }
</style>

<div class="padding-10px">
    <h3>
        This implementation is intended for educational or testing purposes only.</br>
        The <span class="python-function">secp256k1_toy</span> module uses the curve over <b>F₁₇</b> — not real crypto.</br>
        Section 8 derives a real Ethereum address using pure Python (self-contained, no extra libraries).
    </h3>
</div>

### 1️⃣ Introduction

This notebook walks through SECP256k1 — the elliptic curve used by Ethereum and Bitcoin — from first principles.  
We start with a **toy version over F₁₇** to build intuition about point arithmetic and key generation,  
then move to the **actual 256-bit curve** to produce a real Ethereum wallet address entirely in Python.

The pipeline is:

$$
\underbrace{k}_{\text{private key}} \xrightarrow{\times G} \underbrace{Q = (x,y)}_{\text{public key}} \xrightarrow{\text{Keccak-256}} \underbrace{\text{addr}}_{\text{last 20 bytes}}
$$

### 2️⃣ Curve Parameters

SECP256k1 is a **Koblitz curve** — a special case of the Weierstrass form with $a = 0$:

*Weistrass curve*

$$
y^2 \equiv x^3 + a*x + 7 \pmod{p}
$$

*Koblitz curve* $\rightarrow$ ($a = 0, b = 7$)

$$
y^2 \equiv x^3 + 7 \pmod{p}
$$

The curve is defined over a **prime field** $\mathbb{F}_p$, and the key parameters are:

| Parameter | Symbol | Toy (this notebook) | Real SECP256k1 |
|---|---|---|---|
| Field prime | $p$ | $17$ | $2^{256} - 2^{32} - 2^9 - 2^8 - 2^7 - 2^6 - 2^4 - 1$ |
| Curve coefficient | $a$ | $0$ | $0$ |
| Curve coefficient | $b$ | $7$ | $7$ |
| Base point (generator) | $G$ | $(12, 1)$ | $(G_x, G_y)$ — 256-bit coordinates |
| Group order | $n$ | $18$ | $\approx 2^{256}$ |

The **security of ECDSA** rests on the **elliptic curve discrete logarithm problem (ECDLP)**:  
given $Q = kG$, finding $k$ is computationally infeasible classically — but **Shor's algorithm** on a quantum computer can solve it in polynomial time.

### Weistrass Curve

![WEISTRASS](./img/weistrass.png)

### SECP256k1 Curve

![SECP256k1](./img/secp256k1.png)

In [1]:
# Toy curve parameters  —  SECP256k1 family over F_17
P: int = 17       # Field prime (tiny, for illustration)
A: int = 0        # Curve coefficient a  (Koblitz: always 0)
B: int = 7        # Curve coefficient b
G: tuple = (12, 1)  # Base point G
N: int = 18       # Order of G — number of distinct points in <G>

print(f"Curve: y² ≡ x³ + {B}  (mod {P})")
print(f"Base point G = {G}")
print(f"Group order  n = {N}")

Curve: y² ≡ x³ + 7  (mod 17)
Base point G = (12, 1)
Group order  n = 18


### 3️⃣ Point Arithmetic on the Curve

All wallet operations reduce to two geometric constructions:

---

#### Point Addition  $P_1 + P_2$  (chord formula)

Given two distinct points $P_1 = (x_1, y_1)$ and $P_2 = (x_2, y_2)$:

$$
m = \frac{y_2 - y_1}{x_2 - x_1} \pmod{p}
\qquad
x_3 = m^2 - x_1 - x_2 \pmod{p}
\qquad
y_3 = m(x_1 - x_3) - y_1 \pmod{p}
$$

**Special cases:**  
- $P + \mathcal{O} = P$ (identity / point at infinity)  
- $P + (-P) = \mathcal{O}$ (vertical chord)

---

#### Point Doubling  $2P$  (tangent formula)

When $P_1 = P_2 = (x_1, y_1)$:

$$
m = \frac{3x_1^2 + a}{2y_1} \pmod{p}
$$

Then $x_3, y_3$ use the same formulas. Because $a = 0$ on SECP256k1, this simplifies to:

$$
m = \frac{3x_1^2}{2y_1} \pmod{p}
$$

---

#### Scalar Multiplication  $Q = kG$  (double-and-add)

Repeated addition is too slow; instead we write $k$ in binary and scan its bits:

$$
k = \sum_{i=0}^{\lfloor \log_2 k \rfloor} b_i \cdot 2^i
\qquad \Rightarrow \qquad
Q = \sum_{i} b_i \cdot 2^i G
$$

This gives $O(\log k)$ group operations — just ~256 doublings + ~128 additions on average for the real curve.

 > The functions defined below are available as a standalone module in [`secp256k1_toy.py`](../lib/secp256k1_toy.py).

In [2]:
import sys
sys.path.append('..')

In [3]:
from lib.secp256k1_toy import (
    mod_inv, is_on_curve, point_add, point_double,
    scalar_mult, generate_keypair, subgroup_points,
    P, A, B, G, N, INFINITY,
)

### 4️⃣ Verifying the Base Point & Enumerating the Subgroup

In [4]:
# Verify G = (12, 1) satisfies y² ≡ x³ + 7 (mod 17)
x, y = G
lhs = (y * y) % P                  # y²  mod p
rhs = (x * x * x + B) % P         # x³ + 7  mod p
print(f"Curve equation check for G = {G}:")
print(f"  y² mod {P} = {y}² mod {P} = {lhs}")
print(f"  x³ + 7 mod {P} = {x}³ + 7 mod {P} = {rhs}")
print(f"  G is on the curve: {lhs == rhs}")

# Enumerate every point in the cyclic subgroup generated by G
print(f"\nAll {N} points in ⟨G⟩ (including point at infinity):")
for i, pt in enumerate(subgroup_points(), 1):
    label = "∞ (identity)" if pt is INFINITY else str(pt)
    print(f"  {i:2d}·G = {label}")
    if not is_on_curve(pt):
        print("       ← NOT on curve! (bug)")    # safety check

Curve equation check for G = (12, 1):
  y² mod 17 = 1² mod 17 = 1
  x³ + 7 mod 17 = 12³ + 7 mod 17 = 1
  G is on the curve: True

All 18 points in ⟨G⟩ (including point at infinity):
   1·G = (12, 1)
   2·G = (1, 12)
   3·G = (5, 9)
   4·G = (2, 7)
   5·G = (2, 10)
   6·G = (5, 8)
   7·G = (1, 5)
   8·G = (12, 16)
   9·G = ∞ (identity)
  10·G = (12, 1)
  11·G = (1, 12)
  12·G = (5, 9)
  13·G = (2, 7)
  14·G = (2, 10)
  15·G = (5, 8)
  16·G = (1, 5)
  17·G = (12, 16)
  18·G = ∞ (identity)


### 5️⃣ Scalar Multiplication — Double-and-Add

The algorithm processes the binary representation of $k$ from LSB to MSB:

$$
\text{result} \gets \mathcal{O}, \quad \text{addend} \gets G
$$
$$
\text{for each bit } b_i \text{ of } k:
\quad \text{if } b_i = 1 \Rightarrow \text{result} \mathrel{+}= \text{addend}
\quad ; \quad \text{addend} \gets 2 \cdot \text{addend}
$$

At each step the intermediate point must lie on the curve — we verify this below.

In [5]:
def scalar_mult_verbose(k: int, point: tuple, p: int = P, a: int = A) -> tuple:
    """Double-and-add with step-by-step trace output."""
    result = INFINITY
    addend = point
    bits = bin(k)[2:]                          # binary representation of k

    print(f"Computing {k}·G  (k = {k} = 0b{bits})")
    print(f"{'Step':>5}  {'bit':>3}  {'addend':>12}  {'result':>12}")
    print("-" * 50)

    for i, bit in enumerate(reversed(bits)):   # iterate LSB → MSB
        if bit == '1':
            result = point_add(result, addend, p, a)
        addend = point_double(addend, p, a)
        r_label = "∞" if result is INFINITY else str(result)
        a_label = "∞" if addend is INFINITY else str(addend)
        print(f"{i:>5}    {bit}   {a_label:>12}  {r_label:>12}")

    print("-" * 50)
    print(f"Result: {k}·G = {result},  on curve: {is_on_curve(result)}")
    return result

# Trace k = 5 as an example
_ = scalar_mult_verbose(5, G)

Computing 5·G  (k = 5 = 0b101)
 Step  bit        addend        result
--------------------------------------------------
    0    1        (1, 12)       (12, 1)
    1    0         (2, 7)       (12, 1)
    2    1       (12, 16)       (2, 10)
--------------------------------------------------
Result: 5·G = (2, 10),  on curve: True


### 6️⃣ Toy Wallet Generation

An Ethereum (or Bitcoin) wallet is just a number — the **private key** $k$.  
The corresponding **public key** is $Q = kG$, a point on the curve.

$$
\text{Private key: } k \xleftarrow{\text{random}} [1,\, n-1]
$$
$$
\text{Public key:  } Q = k \cdot G = \underbrace{G + G + \cdots + G}_{k \text{ times}}
$$

The private key must stay secret; the public key (and hence the address) can be shared freely.  
The security guarantee: you **cannot** recover $k$ from $Q$ without solving the ECDLP.

In [6]:
# Generate a toy keypair
private_key, public_key = generate_keypair()

print("=" * 40)
print("  TOY SECP256k1 Wallet")
print("=" * 40)
print(f"  Curve:       y² ≡ x³ + {B}  (mod {P})")
print(f"  Group order: n = {N}")
print()
print(f"  Private key  k  = {private_key}")
print(f"  Public key   Q  = {public_key}")
print(f"  Q on curve       : {is_on_curve(public_key)}")
print("=" * 40)

  TOY SECP256k1 Wallet
  Curve:       y² ≡ x³ + 7  (mod 17)
  Group order: n = 18

  Private key  k  = 10
  Public key   Q  = (12, 1)
  Q on curve       : True


### 7️⃣ Step-by-Step Key Generation Visualization

In [7]:
import secrets as _secrets

def generate_keypair_verbose(n: int = N, g: tuple = G, p: int = P, a: int = A) -> tuple:
    """Generate a keypair with a detailed step-by-step trace."""
    # Step 1 — choose private key
    k = 1 + _secrets.randbelow(n - 1)
    print("=" * 50)
    print("  STEP 1 — Choose private key")
    print(f"    k = random ∈ [1, n-1] = [1, {n-1}]")
    print(f"    k = {k}  (binary: {bin(k)})")

    # Step 2 — compute public key via scalar multiplication
    print("\n  STEP 2 — Compute public key  Q = k·G")
    print(f"    Starting from G = {g}")
    Q = INFINITY
    addend = g
    bits = bin(k)[2:]
    for i, bit in enumerate(reversed(bits)):
        old_result = Q
        if bit == '1':
            Q = point_add(Q, addend, p, a)
        double = point_double(addend, p, a)
        if bit == '1':
            q_str = "∞" if Q is INFINITY else str(Q)
            o_str = "∞" if old_result is INFINITY else str(old_result)
            print(f"    bit[{i}]=1 → result: {o_str} + {addend} = {q_str}")
        addend = double

    print("\n  STEP 3 — Public key")
    print(f"    Q = {Q}")
    print(f"    Q ∈ curve: {is_on_curve(Q)}")
    print("=" * 50)
    return k, Q

k_demo, Q_demo = generate_keypair_verbose()

  STEP 1 — Choose private key
    k = random ∈ [1, n-1] = [1, 17]
    k = 9  (binary: 0b1001)

  STEP 2 — Compute public key  Q = k·G
    Starting from G = (12, 1)
    bit[0]=1 → result: ∞ + (12, 1) = (12, 1)
    bit[3]=1 → result: (12, 1) + (12, 16) = ∞

  STEP 3 — Public key
    Q = None
    Q ∈ curve: True


### 8️⃣ Interactive Toy Wallet Widget

In [8]:
from ipywidgets import interact, IntSlider

def interactive_wallet(k: int) -> None:
    """Show what happens when we choose a specific private key k."""
    Q = scalar_mult(k, G)
    q_str = "∞ (identity)" if Q is INFINITY else str(Q)
    valid = is_on_curve(Q)

    print("=" * 45)
    print("  Toy SECP256k1 Wallet Explorer")
    print("=" * 45)
    print(f"  Private key  k = {k}")
    print(f"  Public key   Q = k·G = {q_str}")
    print(f"  Q on curve       : {valid}")

    if Q is not INFINITY:
        print(f"\n  Q.x = {Q[0]}")
        print(f"  Q.y = {Q[1]}")

    # Show how Q relates to the full subgroup table
    pts = subgroup_points()
    idx = pts.index(Q) if Q in pts else -1
    if idx != -1:
        print(f"\n  Q is the {idx+1}-th point in ⟨G⟩")
    print("=" * 45)

interact(
    interactive_wallet,
    k=IntSlider(min=1, max=N-1, step=1, value=5, description='k (private key)')
)

interactive(children=(IntSlider(value=5, description='k (private key)', max=17, min=1), Output()), _dom_classe…

<function __main__.interactive_wallet(k: int) -> None>

### 9️⃣ Real SECP256k1

<style>
    .python-function {
        color: #228B22;
        font-family: monospace;
    }
</style>

Full 256-bit prime field, 256-bit group order, 256-bit coordinates.</br>
This is the algorithm Ethereum uses for all accounts — every address you've ever seen starts here.</br>
The implementation below is **pure Python**, zero extra dependencies.

The real curve parameters are:

| Parameter | Value |
|---|---|
| $p$ | `0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F` |
| $G_x$ | `0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798` |
| $G_y$ | `0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8` |
| $n$   | `0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141` |

In [9]:
# Real SECP256k1 domain parameters  (FIPS / SEC2)
REAL_P  = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
REAL_A  = 0
REAL_B  = 7
REAL_Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
REAL_Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
REAL_G  = (REAL_Gx, REAL_Gy)
REAL_N  = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

print(f"Field prime p   (hex): ...{hex(REAL_P)[-8:]}")
print(f"Generator  Gx  (hex): ...{hex(REAL_Gx)[-8:]}")
print(f"Generator  Gy  (hex): ...{hex(REAL_Gy)[-8:]}")
print(f"Group order n  (hex): ...{hex(REAL_N)[-8:]}")
print(f"\np bit-length:  {REAL_P.bit_length()}")
print(f"n bit-length:  {REAL_N.bit_length()}")

Field prime p   (hex): ...fffffc2f
Generator  Gx  (hex): ...16f81798
Generator  Gy  (hex): ...fb10d4b8
Group order n  (hex): ...d0364141

p bit-length:  256
n bit-length:  256


#### The Ethereum Address Derivation Pipeline

$$
k \xrightarrow{Q = kG} (Q_x, Q_y) \xrightarrow{\text{uncompressed pubkey}} \underbrace{04 \| Q_x \| Q_y}_{65\,\text{bytes}} \xrightarrow{\text{Keccak-256}} \underbrace{H}_{32\,\text{bytes}} \xrightarrow{\text{last 20 bytes}} \underbrace{\text{addr}}_{20\,\text{bytes}}
$$

Steps:
1. Generate random 256-bit private key $k \in [1, n-1]$
2. Compute uncompressed public key: `0x04 ‖ Qx (32 bytes) ‖ Qy (32 bytes)` — 65 bytes total
3. Hash the **64 bytes of Qx ‖ Qy** (no prefix) with Keccak-256 → 32 bytes
4. Take the **last 20 bytes** → Ethereum address (optionally EIP-55 checksum-encoded)

In [10]:
import secrets

# ── Real SECP256k1 point arithmetic (reuses the same chord/tangent formulas) ──

def real_mod_inv(a: int) -> int:
    """Modular inverse mod REAL_P via Fermat's little theorem."""
    return pow(a, REAL_P - 2, REAL_P)

def real_point_add(P1, P2):
    """Elliptic-curve point addition over the real SECP256k1 field."""
    if P1 is None:
        return P2
    if P2 is None:
        return P1
    x1, y1 = P1
    x2, y2 = P2
    if x1 == x2:
        if y1 == y2:
            return real_point_double(P1)
        return None  # P + (-P) = ∞
    m = (y2 - y1) * real_mod_inv(x2 - x1) % REAL_P
    x3 = (m * m - x1 - x2) % REAL_P
    y3 = (m * (x1 - x3) - y1) % REAL_P
    return (x3, y3)

def real_point_double(P1):
    """Elliptic-curve point doubling over the real SECP256k1 field."""
    if P1 is None:
        return None
    x1, y1 = P1
    if y1 == 0:
        return None
    # a = 0, so m = 3x² / 2y
    m = (3 * x1 * x1) * real_mod_inv(2 * y1) % REAL_P
    x3 = (m * m - 2 * x1) % REAL_P
    y3 = (m * (x1 - x3) - y1) % REAL_P
    return (x3, y3)

def real_scalar_mult(k: int, point: tuple) -> tuple:
    """Double-and-add scalar multiplication on the real SECP256k1 curve."""
    result = None
    addend = point
    while k:
        if k & 1:
            result = real_point_add(result, addend)
        addend = real_point_double(addend)
        k >>= 1
    return result

print("Real SECP256k1 arithmetic loaded.")
# Quick sanity check: n·G should equal the point at infinity
nG = real_scalar_mult(REAL_N, REAL_G)
print(f"n·G = {'∞ ✓' if nG is None else nG}  (expected: ∞)")

Real SECP256k1 arithmetic loaded.
n·G = ∞ ✓  (expected: ∞)


#### Keccak-256 — Pure Python (reused from `keccak.ipynb`)

We need Keccak-256 to hash the public key. The implementation from `keccak.ipynb` section 8 is reused here verbatim — it passes all official test vectors.

In [11]:
# ── Keccak-256 (pure Python, zero dependencies) ─────────────────────────────

# 24 round constants (LFSR-derived)
_KECCAK_RC = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008,
]

# ρ rotation offsets (5×5 lane table)
_KECCAK_RHO = [
    [ 0, 36,  3, 41, 18],
    [ 1, 44, 10, 45,  2],
    [62,  6, 43, 15, 61],
    [28, 55, 25, 21, 56],
    [27, 20, 39,  8, 14],
]

def _rot64(x: int, n: int) -> int:
    return ((x << n) | (x >> (64 - n))) & 0xFFFFFFFFFFFFFFFF

def _keccak_f1600(state: list) -> list:
    for rc in _KECCAK_RC:
        # θ — column parity diffusion
        C = [state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4] for x in range(5)]
        D = [C[(x - 1) % 5] ^ _rot64(C[(x + 1) % 5], 1) for x in range(5)]
        state = [[state[x][y] ^ D[x] for y in range(5)] for x in range(5)]
        # ρ + π — rotation + permutation
        B = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                B[y][(2 * x + 3 * y) % 5] = _rot64(state[x][y], _KECCAK_RHO[x][y])
        # χ — non-linear mixing (same structure as toy Keccak)
        state = [[B[x][y] ^ ((~B[(x + 1) % 5][y]) & B[(x + 2) % 5][y]) for y in range(5)] for x in range(5)]
        # ι — round constant injection
        state[0][0] ^= rc
    return state

def keccak256(message: bytes) -> bytes:
    """
    Keccak-256 hash (Ethereum variant — padding 0x01, NOT SHA3's 0x06).
    Returns 32 raw bytes.
    """
    rate_bytes = 136   # r = 1088 bits
    msg = bytearray(message)
    # 10*1 padding (Keccak standard)
    msg.append(0x01)
    while len(msg) % rate_bytes != 0:
        msg.append(0x00)
    msg[-1] |= 0x80

    state = [[0] * 5 for _ in range(5)]

    # Absorb
    for block_start in range(0, len(msg), rate_bytes):
        block = msg[block_start:block_start + rate_bytes]
        for i in range(rate_bytes // 8):
            x, y = i % 5, i // 5
            lane = int.from_bytes(block[i * 8:(i + 1) * 8], 'little')
            state[x][y] ^= lane
        state = _keccak_f1600(state)

    # Squeeze — first 256 bits (4 lanes)
    output = bytearray()
    for i in range(4):
        output += state[i % 5][i // 5].to_bytes(8, 'little')
    return bytes(output)

# Verify test vectors
assert keccak256(b'').hex() == 'c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470'
assert keccak256(b'abc').hex() == '4e03657aea45a94fc7d47ba826c8d667c0d1e6e33a64a036ec44f58fa12d6c45'
print("✓ Keccak-256 test vectors pass")

✓ Keccak-256 test vectors pass


#### EIP-55 Checksum Encoding

Raw Ethereum addresses are 20 bytes in lowercase hex.  
**EIP-55** encodes a checksum *inside the capitalisation* of the hex characters:

[EIP-55 Link](https://eips.ethereum.org/EIPS/eip-55)

$$
\text{addr}[i] = \begin{cases} \text{uppercase} & \text{if } \text{keccak256}(\text{addr\_lower})[i] \geq 8 \\ \text{lowercase} & \text{otherwise} \end{cases}
$$

This lets wallets silently detect accidental character substitutions.

In [12]:
def eip55_checksum(address_hex: str) -> str:
    """
    Apply EIP-55 mixed-case checksum to a 40-char hex address string.
    Input:  '0x' + 40 hex chars  OR  raw 40 hex chars (lowercase)
    Output: '0x' + checksum-encoded 40 hex chars
    """
    addr = address_hex.lower().removeprefix('0x')
    h = keccak256(addr.encode()).hex()
    out = ''.join(
        c.upper() if int(h[i], 16) >= 8 else c
        for i, c in enumerate(addr)
    )
    return '0x' + out


def pubkey_to_address(Qx: int, Qy: int) -> str:
    """
    Derive an Ethereum address from an uncompressed SECP256k1 public key.

    Algorithm:
      1. Serialise Qx ‖ Qy as 64 bytes (32 big-endian each)
      2. Keccak-256 hash those 64 bytes → 32-byte digest
      3. Take the last 20 bytes → raw address
      4. Apply EIP-55 checksum
    """
    pub_bytes = Qx.to_bytes(32, 'big') + Qy.to_bytes(32, 'big')   # 64 bytes, no 0x04 prefix
    digest    = keccak256(pub_bytes)                                # 32-byte hash
    raw_addr  = digest[-20:].hex()                                  # last 20 bytes = 40 hex chars
    return eip55_checksum(raw_addr)

print("Address derivation functions loaded.")

Address derivation functions loaded.


#### Generating a Real Ethereum Wallet

In [13]:
# Step 1 — Generate random private key  k ∈ [1, n-1]
real_k = 1 + secrets.randbelow(REAL_N - 1)

# Step 2 — Compute public key  Q = k·G  on the real 256-bit curve
real_Q = real_scalar_mult(real_k, REAL_G)
real_Qx, real_Qy = real_Q

# Step 3 — Derive Ethereum address
eth_address = pubkey_to_address(real_Qx, real_Qy)

print("=" * 72)
print("  REAL SECP256k1 Ethereum Wallet")
print("=" * 72)
print()
print("  Private key (hex, 32 bytes):")
print(f"    0x{real_k:064x}")
print()
print("  Public key — uncompressed (65 bytes):")
print("    04")
print(f"    Qx: 0x{real_Qx:064x}")
print(f"    Qy: 0x{real_Qy:064x}")
print()
print("  Public key — compressed (33 bytes):")
prefix = "02" if real_Qy % 2 == 0 else "03"
print(f"    {prefix}{real_Qx:064x}")
print()
print("  Ethereum Address (EIP-55):")
print(f"    {eth_address}")
print("=" * 72)

  REAL SECP256k1 Ethereum Wallet

  Private key (hex, 32 bytes):
    0x0dd6d4b8503e6e69989da7f5215ee5357c50816af79f2f207ab653784eb2c427

  Public key — uncompressed (65 bytes):
    04
    Qx: 0x5ca5f6247e22ed633e4aded3cb12c1d272168955c3259ddc8323eafe258fb969
    Qy: 0x9b94a0827c04935932be04192f539a12868ff095a9e097572e72a46f0eb92fdb

  Public key — compressed (33 bytes):
    035ca5f6247e22ed633e4aded3cb12c1d272168955c3259ddc8323eafe258fb969

  Ethereum Address (EIP-55):
    0x1f8cC701C7f4165f4CD880e25f6f5ac180c8BFf1


#### Known Test Vector Verification

We verify against the well-known test vector from the Ethereum Yellow Paper annex and community references.  
Private key `0x01` must produce a specific, publicly documented address.

In [ ]:
# ── Test vector: private key = 1 ─────────────────────────────────────────────
# G itself is the public key when k = 1
test_Q = real_scalar_mult(1, REAL_G)

# Sanity: 1·G must equal G
assert test_Q == REAL_G, "1·G should equal G"

# Derive address for k = 1
tv_addr = pubkey_to_address(test_Q[0], test_Q[1])

print("k = 1")
print(f"Q = G = ({hex(REAL_Gx)[:10]}..., {hex(REAL_Gy)[:10]}...)")
print(f"Address: {tv_addr}")
print()

# ── Test vector: well-known private key ──────────────────────────────────────
# From Mastering Ethereum (Antonopoulos) and various community test suites.
# private key = 0xf8f8a2f43c8376ccb0871305060d7b27b0554d2cc72bccf41b2705608452f315
KNOWN_K    = 0xf8f8a2f43c8376ccb0871305060d7b27b0554d2cc72bccf41b2705608452f315
KNOWN_ADDR = "0x001d3F1ef827552Ae1114027BD3ECF1f086bA0F9"

known_Q   = real_scalar_mult(KNOWN_K, REAL_G)
known_Qx, known_Qy = known_Q
derived   = pubkey_to_address(known_Qx, known_Qy)
match     = derived.lower() == KNOWN_ADDR.lower()

print("Known test vector (Mastering Ethereum):")
print(f"  k   = 0x{KNOWN_K:064x}")
print(f"  Qx  = 0x{known_Qx:064x}")
print(f"  Qy  = 0x{known_Qy:064x}")
print(f"  Expected: {KNOWN_ADDR}")
print(f"  Derived:  {derived}")
print(f"  Match: {'✓' if match else '✗  ← FAIL'}")

k = 1
Q = G = (0x79be667e..., 0x483ada77...)
Address: 0x7E5F4552091A69125d5DfCb7b8C2659029395Bdf

Known test vector (Mastering Ethereum):
  k   = 0xf8f8a2f43c8376ccb0871305060d7b27b0554d2cc72bccf41b2705608452f315
  Qx  = 0x6e145ccef1033dea239875dd00dfb4fee6e3348b84985c92f103444683bae07b
  Qy  = 0x83b5c38e5e2b0c8529d7fa3f64d46daa1ece2d9ac14cab9477d042c84c32ccd0
  Expected: 0x001d3F1ef827552Ae1114027BD3ECF1f086bA0F9
  Derived:  0x001d3F1ef827552Ae1114027BD3ECF1f086bA0F9
  Match: ✓


### 🔟 Notes on Quantum Hardness

- **Classical security**: The best known algorithm (Baby-step Giant-step / Pollard's rho) runs in $O(\sqrt{n})$ time — roughly $2^{128}$ operations for SECP256k1. Infeasible.

- **Quantum threat — Shor's algorithm**: A sufficiently large quantum computer can solve the ECDLP in $O(\log^3 n)$ operations using the **quantum period-finding** subroutine. For SECP256k1 this requires ~2330 logical qubits with full error correction, far beyond current hardware.

- **Grover's algorithm**: Provides only a quadratic speedup for symmetric primitives (Keccak-256 in this context). The address-mining search space is $2^{160}$, reduced to $2^{80}$ by Grover — still infeasible in practice, but this is why AES-256 / SHA3-512 are recommended for post-quantum symmetric security.

- **The toy curve (F₁₇)**: With only $n = 18$ points, Grover or Shor become trivially feasible — the entire purpose of the toy is to make the *structure* of the ECDLP observable by hand.

| Metric | Toy (F₁₇) | Real SECP256k1 |
|---|---|---|
| Group order $n$ | $18$ | $\approx 2^{256}$ |
| Classical ECDLP | brute-force in $< 1$ μs | $\approx 2^{128}$ ops |
| Shor's algorithm | a few gates | $\sim 2330$ logical qubits |
| Grover on address | trivial | $\approx 2^{80}$ ops |